# Week 3, Lecture 2: EDA practical
## Bike-rental demand: Student notebook

This practical uses a different scenario from the lecture's breast-cancer examples.
Your task is to explore daily bike rentals and the weather conditions recorded on each day.
Complete **Setup and Exercises 01–10 in 60 minutes**. Choose an optional challenge if you finish early.

### Exercise-to-slide map

The numbers below are the **1-based PowerPoint slide numbers, including the title slide**, in
`Week03_Lecture02_EDA_PartB(1).pptx` (24 slides). Slides 1–2 introduce the session.

| Exercise | Practical task | Exact related lecture slides | Minutes |
|---|---|---|---:|
| Setup | Install packages, download and understand the data | 4–6: dataset context and records | 5 |
| Exercise 01 | Automated EDA with Sweetviz | 3: Automated EDA | 4 |
| Exercise 02 | Dataset structure and numerical summaries | 4–6: dataset context and first records; 7–10: summaries and interpretation | 7 |
| Exercise 03 | Categorical frequencies and a pie chart | 11–13: visualisation and categorical pie chart | 4 |
| Exercise 04 | A numerical distribution with a histogram | 12: plotting functions; 14: histogram | 4 |
| Exercise 05 | An aggregated time-series line chart | 12: plotting functions; 15: line chart | 5 |
| Exercise 06 | Compare rental distributions with boxplots | 12: plotting functions; 16 and 23: numerical comparisons by category | 5 |
| Exercise 07 | Find and inspect potential outliers | 17–20: finding and interpreting outliers | 7 |
| Exercise 08 | Compare an outlier-handling policy | 21: fixing outliers | 5 |
| Exercise 09 | Correlation analysis and a scatter plot | 12: scatter plots; 22: correlation heatmap | 8 |
| Exercise 10 | Encoding and preprocessing decisions | 24: data types, encoding, imbalance and scaling | 6 |
| **Core total** | **Setup and Exercises 01–10** | | **60** |
| Optional A | Pairplots and rank correlation | 12: pairplots; 22: relationships | 8–10 extra |
| Optional B | Compare outlier policies within weather groups | 17–21: outlier detection and handling | 10 extra |
| Optional C | Repair deliberately introduced data-quality issues | 7–10: missing values and types; 24: preprocessing | 10 extra |
| Optional D | A pipeline with a chronological split and leakage checks | 24: encoding and scaling, extended to a predictive model | 15 extra |

The TODO cells are scaffolded so Run all completes before you fill them. Complete each TODO and rerun that cell, then write a short interpretation. Worked examples and setup code are already provided.


## Scenario and dataset

You are helping a bike-sharing operator understand demand before building a model to predict **total daily rentals**.
Questions include whether demand varies across weather conditions, how it changes over time,
and which observations deserve investigation.

The **UCI Bike Sharing dataset** contains Capital Bikeshare records from 2011–2012.
We download its official archive and use **`day.csv`: 731 days and 16 original columns**.
No account, API key, manual file upload or separate dataset download is required.
`cnt` is the daily total, with `cnt = casual + registered`.

Source: [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset),
Hadi Fanaee-T and João Gama, [DOI: 10.24432/C5W894](https://doi.org/10.24432/C5W894).
The source lists a CC BY 4.0 licence. The source data remain unchanged.

### Google Colab

Upload this `.ipynb` to Colab. Use **Runtime > Run all**, or run cells in order.
Use a standard CPU runtime. Python, pandas, NumPy, Matplotlib, seaborn and scikit-learn
are normally available in Colab. The next cell installs the additional **Sweetviz** package
and ensures the other packages used here are available. Sweetviz is pinned to the tested version.
Installation and the first download need internet access.


## Setup (5 minutes)
Run these cells before Exercise 01.


In [ ]:
%pip -q install "sweetviz==2.3.3" "pandas>=1.5" "numpy>=1.23" "matplotlib>=3.10" "seaborn>=0.12" "scikit-learn>=1.2"


In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile, BadZipFile
from urllib.request import urlopen
from urllib.error import URLError
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import sweetviz as sv
from IPython.display import display, FileLink
from importlib.metadata import version

%matplotlib inline
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.05)
BLUE = "#2676AF"
TEAL = "#128B83"
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 3)

print("Python:", sys.version.split()[0])
for package in ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "sweetviz"]:
    print(f"{package}: {version(package)}")


In [ ]:
# Both download locations belong to UCI. The second is the older official URL.
DATA_URLS = [
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip",
]
csv_path = Path("bike_sharing_day.csv")

if csv_path.exists():
    raw = pd.read_csv(csv_path)
    print("Using the CSV already downloaded in this runtime.")
else:
    errors = []
    for url in DATA_URLS:
        try:
            with urlopen(url, timeout=30) as response:
                archive_bytes = response.read()
            with ZipFile(BytesIO(archive_bytes)) as archive:
                day_file = next(name for name in archive.namelist() if name.endswith("day.csv"))
                csv_bytes = archive.read(day_file)
            csv_path.write_bytes(csv_bytes)
            raw = pd.read_csv(BytesIO(csv_bytes))
            print("Downloaded the official UCI daily dataset.")
            break
        except (URLError, TimeoutError, OSError, BadZipFile, StopIteration) as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
    else:
        raise RuntimeError("UCI download failed. Check your connection and rerun this cell. " + " / ".join(errors))

required_columns = {"instant", "dteday", "season", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed", "casual", "registered", "cnt"}
assert required_columns.issubset(raw.columns), "Unexpected CSV schema. Remove the cached CSV and rerun."
assert len(raw) == 731, "This practical expects the original 731-row daily file."
assert (raw["cnt"] == raw["casual"] + raw["registered"]).all()
print(f"Original data: {raw.shape[0]} rows, {raw.shape[1]} columns.")
display(raw.head())


In [ ]:
# Keep raw intact. Give the practical columns readable names and semantic types.
season_map = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Autumn"}
weather_map = {1: "Clear/partly cloudy", 2: "Misty/cloudy", 3: "Light rain/snow", 4: "Heavy rain/snow"}
day_type_map = {0: "Weekend/holiday", 1: "Working day"}

df = pd.DataFrame({
    "record_id": raw["instant"],
    "date": pd.to_datetime(raw["dteday"]),
    "season": raw["season"].map(season_map),
    "weather": raw["weathersit"].map(weather_map),
    "day_type": raw["workingday"].map(day_type_map),
    "temp": raw["temp"],
    "feels_like_temp": raw["atemp"],
    "humidity_pct": 100 * raw["hum"],
    "windspeed": raw["windspeed"],
    "rentals": raw["cnt"],
    "casual": raw["casual"],
    "registered": raw["registered"],
})
df["year"] = df["date"].dt.year
for col, categories in {
    "season": list(season_map.values()),
    "weather": list(weather_map.values()),
    "day_type": list(day_type_map.values()),
}.items():
    df[col] = pd.Categorical(df[col], categories=categories)
df = df.sort_values("date").reset_index(drop=True)
assert df.shape == (731, 13)
assert df[["season", "weather", "day_type"]].notna().all().all()
measure_cols = ["temp", "feels_like_temp", "humidity_pct", "windspeed", "rentals"]
raw_snapshot = raw.copy(deep=True)
print("Practical data:", df.shape)
display(df.head())

def iqr_bounds(series):
    """Return observed Q1, Q3 and the conventional 1.5 × IQR fences."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1, q3, q1 - 1.5 * iqr, q3 + 1.5 * iqr


### Variables used in the practical

| Practical column | Meaning and role |
|---|---|
| `date` | Calendar date. Use it to put observations in time order. |
| `record_id` | Row identifier from `instant`. It is not a measurement. |
| `season` | Unordered categories: Winter, Spring, Summer, Autumn. The source coding is 1–4 respectively. |
| `weather` | A category describing the day's weather. No heavy-weather days occur in this daily file. |
| `day_type` | Working day or weekend/holiday. |
| `temp`, `feels_like_temp` | Normalised temperature measures from the source, used as unitless values here. |
| `humidity_pct` | Relative humidity in %, obtained from `100 × hum`. |
| `windspeed` | Source normalised wind speed, used as a unitless value here. |
| `rentals` | Total rentals that day: the numerical count target for a regression task. |
| `casual`, `registered` | Same-day components of `rentals`. They are unavailable when forecasting the total in advance. |
| `year` | Calendar year, derived from the date. |

Use variable meaning as well as storage type and unique-value counts when choosing an EDA method.
Weather codes are categories even though the source stores them as integers.


## Exercise 01: Automated EDA

**Related slides:** 3: Automated EDA. **Time:** 4 minutes.

Run the worked code. It creates a Sweetviz HTML report with `rentals` as the numerical target.
Inspect the overview and one numerical feature. Record two observations to verify manually later.
Which temperature features appear to contain similar information? Does a report replace your interpretation?


In [ ]:
# Sweetviz works best here with plain categorical strings and numerical measures.
report_df = df[["season", "weather", "day_type"] + measure_cols].copy()
for col in ["season", "weather", "day_type"]:
    report_df[col] = report_df[col].astype(str)
config = sv.FeatureConfig(force_cat=["season", "weather", "day_type"], force_num=measure_cols)
report = sv.analyze(report_df, target_feat="rentals", feat_cfg=config, pairwise_analysis="on")
report_path = Path("bike_sharing_sweetviz.html")
report.show_html(str(report_path), open_browser=False)
report.show_notebook(w="100%", h=650, scale=0.75)


In [ ]:
# Open this file from Colab's Files panel if you want the full report.
print("Report saved to:", report_path.resolve())
display(FileLink(str(report_path)))


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 02: Structure and summary statistics

**Related slides:** 4–6: dataset context and first records; 7–10: summaries and interpretation. **Time:** 7 minutes.

The worked example shows `shape`, `head`, `info` and numerical `describe` output.
Complete the feature overview with **non-missing counts, missing counts, unique-value counts and dtypes**.
Which columns are IDs, categories, dates and measurements?
Give the five-number summary for `rentals`. Is `describe()` limited to those five numbers?


In [ ]:
print("Shape:", df.shape)
display(df.head())
df.info()
display(df[measure_cols].describe().T)


In [ ]:
# TODO: build the per-column overview with count(), isna().sum(), nunique() and dtypes.
overview = None
if overview is None:
    print("Complete the overview TODO, then rerun this cell.")
else:
    display(overview)
    print("Five-number summary for daily rentals:")
    display(df["rentals"].quantile([0, .25, .50, .75, 1]).rename("rentals"))


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 03: Categorical frequencies and a pie chart

**Related slides:** 11–13: visualisation and categorical pie chart. **Time:** 4 minutes.

Count the days in each `weather` category and use those counts to complete the pie chart.
The plot omits categories with zero observations, but the count table retains them.
Which category has the weakest coverage? Would weather-category imbalance be the same issue as target-class imbalance when predicting `rentals`?


In [ ]:
# TODO: use value_counts(sort=False) to preserve the declared category order.
weather_counts = None
if weather_counts is None:
    print("Complete the weather-counts TODO.")
else:
    display(pd.DataFrame({"days": weather_counts, "percent": 100 * weather_counts / len(df)}).round(2))
    nonzero = weather_counts[weather_counts > 0]
    labels = [f"{label}\n{count} days" for label, count in nonzero.items()]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.pie(nonzero, labels=labels, autopct="%1.1f%%", startangle=90,
           colors=["#58A7C9", "#E8A064", "#9C85BD"],
           wedgeprops={"edgecolor": "white", "linewidth": 1.5})
    ax.set_title("Days recorded in each weather category")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 04: A histogram of daily rentals

**Related slides:** 12: plotting functions; 14: histogram. **Time:** 4 minutes.

The worked histogram shows normalised wind speed. Complete a histogram of daily `rentals` with 15 bins.
Describe the concentration and range. Change the bin count to 5 or 30 and compare the apparent shape.
Does a bar show a single rental total or a range of totals?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
sns.histplot(data=df, x="windspeed", bins=15, color=TEAL, ax=ax)
ax.set(title="Worked example: normalised wind speed", xlabel="Normalised wind speed", ylabel="Number of days")
plt.tight_layout()
plt.show()


In [ ]:
# TODO: provide the rentals Series.
hist_values = None
if hist_values is None:
    print("Complete the histogram TODO.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(hist_values, bins=15, color=BLUE, edgecolor="white")
    ax.set(title="Distribution of daily bike rentals", xlabel="Total daily rentals", ylabel="Number of days")
    plt.tight_layout()
    plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 05: A line chart with a meaningful time order

**Related slides:** 12: plotting functions; 15: line chart. **Time:** 5 minutes.

Use `date` to calculate the **mean daily rentals for each calendar month**.
Plot the monthly series and label it correctly. Use `resample("MS")` after putting `date` in the index.
What broad pattern is visible? Why can connecting rows in an arbitrary order mislead?


In [ ]:
# TODO: set the date index, select rentals, resample to month starts and take the mean.
monthly_mean = None
if monthly_mean is None:
    print("Complete the monthly-mean TODO.")
else:
    display(monthly_mean.head())
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(monthly_mean.index, monthly_mean.values, marker="o", color=BLUE)
    ax.set(title="Monthly mean of daily rentals", xlabel="Month", ylabel="Mean rentals per day")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 06: Side-by-side boxplots

**Related slides:** 12: plotting functions; 16 and 23: numerical comparisons by category. **Time:** 5 minutes.

Compare daily rentals on working days and weekends/holidays.
The boxplots are provided. Complete a table containing the number of days, mean and median in each group.
Describe overlap and spread as well as typical values. Would the graph alone establish a day-type effect?


In [ ]:
# TODO: group by day_type and summarise rentals with count, mean and median.
day_summary = None
if day_summary is None:
    print("Complete the grouped-summary TODO.")
else:
    display(day_summary.round(2))
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="day_type", y="rentals", order=list(day_type_map.values()),
            color="#80B8D1", width=.5, fliersize=3, ax=ax)
ax.set(title="Daily rentals by day type", xlabel="Day type", ylabel="Total daily rentals")
plt.tight_layout()
plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 07: Potential outliers and the IQR rule

**Related slides:** 17–20: finding and interpreting outliers. **Time:** 7 minutes.

The worked panel gives a boxplot for each numerical measure, excluding IDs and categorical codes.
Use the supplied `iqr_bounds` helper to calculate the wind-speed fences.
Complete the Boolean mask for values below the lower fence or above the upper fence.
Inspect the flagged dates, weather and rentals. Are all flagged observations necessarily incorrect?


In [ ]:
fig, axes = plt.subplots(1, len(measure_cols), figsize=(14, 3.5))
for ax, col in zip(axes, measure_cols):
    ax.boxplot(df[col].dropna(), whis=1.5, widths=.5)
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
fig.suptitle("Worked example: inspect numerical measures", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
q1, q3, lower, upper = iqr_bounds(df["windspeed"])
print(f"Q1={q1:.5f}; Q3={q3:.5f}; IQR={q3-q1:.5f}")
print(f"IQR fences: {lower:.5f} to {upper:.5f}")
# TODO: combine the two comparisons with | and put each comparison in parentheses.
wind_flags = None
if wind_flags is None:
    print("Complete the wind-speed flag mask.")
else:
    print("Potential wind-speed outliers:", int(wind_flags.sum()))
    display(df.loc[wind_flags, ["date", "weather", "windspeed", "rentals"]].sort_values("windspeed", ascending=False))
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.boxplot(df["windspeed"], orientation="horizontal", whis=1.5, widths=.4)
    ax.axvline(upper, color="crimson", linestyle="--", label="Upper IQR fence")
    ax.set(title="Wind-speed observations and the IQR rule", xlabel="Normalised wind speed", yticks=[])
    ax.legend()
    plt.tight_layout()
    plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 08: A capping experiment that preserves the source

**Related slides:** 21: fixing outliers. **Time:** 5 minutes.

Try IQR capping as a **policy experiment** for normalised wind speed.
Complete `Series.clip` with the lower and upper fences. Keep the result separate from `df`.
Compare the original and capped boxplots and the mean, median and maximum.
Explain one reason to keep the original observations instead. Would a cleaner-looking plot prove better data?


In [ ]:
_, _, lower, upper = iqr_bounds(df["windspeed"])
# TODO: clip the wind-speed Series to the IQR fences.
wind_capped = None
if wind_capped is None:
    print("Complete the capping TODO.")
else:
    comparison = pd.DataFrame({"Original": df["windspeed"], "Capped copy": wind_capped})
    display(comparison.agg(["count", "mean", "median", "max"]).T.round(5))
    print("Changed observations:", int((wind_capped != df["windspeed"]).sum()))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharex=True)
    for ax, col in zip(axes, comparison.columns):
        ax.boxplot(comparison[col], orientation="horizontal", whis=1.5, widths=.4)
        ax.set(title=col, xlabel="Normalised wind speed", yticks=[])
    plt.tight_layout()
    plt.show()
    assert df["windspeed"].equals(raw_snapshot["windspeed"].rename("windspeed"))


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 09: Relationships between numerical variables

**Related slides:** 12: scatter plots; 22: correlation heatmap. **Time:** 8 minutes.

Complete a Pearson correlation matrix for `measure_cols`.
The cell draws a heatmap and a scatter plot of normalised temperature versus daily rentals.
Identify the strongest correlation between two input measurements and describe the relationship with rentals.
Why should IDs, nominal category codes and same-day rental components be excluded from this exploratory predictor matrix?


In [ ]:
# TODO: compute the numerical correlation matrix.
corr_matrix = None
if corr_matrix is None:
    print("Complete the correlation TODO.")
else:
    display(corr_matrix.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                square=True, cbar_kws={"label": "Pearson r"}, ax=axes[0])
    axes[0].set_title("Selected numerical correlations")
    axes[1].scatter(df["temp"], df["rentals"], color=BLUE, alpha=.5, s=18)
    axes[1].set(title="Temperature and daily rentals", xlabel="Normalised temperature", ylabel="Total daily rentals")
    plt.tight_layout()
    plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Exercise 10: A preprocessing plan

**Related slides:** 24: data types, encoding, imbalance and scaling. **Time:** 6 minutes.

Choose predictors that could be used with calendar information and weather estimates to predict daily rentals.
Complete one-hot encoding of the three categorical columns in the supplied feature frame.
Answer briefly: why exclude `record_id`, `casual` and `registered`? When would numerical scaling be useful?
Is target-class resampling appropriate for this regression target? Would you use target encoding here?


In [ ]:
feature_cols = ["temp", "humidity_pct", "windspeed", "season", "weather", "day_type"]
feature_frame = df[feature_cols].copy()
# TODO: encode season, weather and day_type with get_dummies(..., dtype=int).
encoded = None
if encoded is None:
    print("Complete the encoding TODO.")
else:
    print("Original predictor shape:", feature_frame.shape)
    print("Encoded predictor shape:", encoded.shape)
    display(encoded.head())

print("Write a preprocessing plan in the interpretation cell.")
print("This encoding cell is an EDA demonstration. Optional D fits preprocessing on the training period only.")


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## End of the 60-minute core route

Before moving on, summarise one finding about the data and one decision about a future model.
Check that your conclusions refer to associations, sample sizes and the available measurements.

**If you finish early:** choose any optional challenge below. They are independent of the core TODO results
and require the shared setup only. They are extensions beyond the 60-minute core route.


## Optional challenge A: Pairplots and Pearson versus Spearman

**Related slides:** 12: pairplots; 22: relationships. **Time:** 8–10 minutes extra.

Choose `temp`, `humidity_pct`, `windspeed` and `rentals` for a pairplot.
Complete the selected column list and compare Pearson and Spearman correlations with rentals.
Use one blue colour for all scatter points. Describe a relationship that a single coefficient may miss.
Explain when rank correlation could be more useful than Pearson correlation.


In [ ]:
# TODO: select the four numerical columns named in the prompt.
pair_cols = None
if pair_cols is None:
    print("Complete the pairplot-column TODO.")
else:
    grid = sns.pairplot(df[pair_cols], diag_kind="hist",
                        plot_kws={"color": BLUE, "alpha": .4, "s": 14},
                        diag_kws={"color": BLUE})
    grid.fig.suptitle("Pairwise numerical relationships", y=1.02)
    plt.show()
    associations = pd.DataFrame({
        "Pearson": df[pair_cols].corr(method="pearson")["rentals"],
        "Spearman": df[pair_cols].corr(method="spearman")["rentals"],
    }).drop(index="rentals")
    display(associations.round(3))


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Optional challenge B: Outlier policies within weather groups

**Related slides:** 17–21: outlier detection and handling. **Time:** 10 minutes extra.

Compare IQR fences from all days with fences estimated separately within each weather group.
Complete the grouped Q1 and Q3 calculations using `groupby(...).transform(...)`.
Compare flag counts, flag percentages and average wind speeds after global versus group-specific capping.
How does the small light-rain/snow sample affect your confidence in its quartiles? Which policy would you defend, and why?


In [ ]:
policy = df[["weather", "windspeed"]].copy()
_, _, global_lower, global_upper = iqr_bounds(policy["windspeed"])
policy["global_flag"] = (policy["windspeed"] < global_lower) | (policy["windspeed"] > global_upper)
# TODO: calculate the within-weather quartiles for every row using transform.
group_q1 = None
group_q3 = None
if group_q1 is None or group_q3 is None:
    print("Complete the two grouped-quartile TODOs.")
else:
    group_iqr = group_q3 - group_q1
    group_lower = group_q1 - 1.5 * group_iqr
    group_upper = group_q3 + 1.5 * group_iqr
    policy["group_flag"] = (policy["windspeed"] < group_lower) | (policy["windspeed"] > group_upper)
    policy["global_capped"] = policy["windspeed"].clip(global_lower, global_upper)
    policy["group_capped"] = policy["windspeed"].clip(lower=group_lower, upper=group_upper)
    policy_summary = policy.groupby("weather", observed=True).agg(
        days=("windspeed", "size"), global_flags=("global_flag", "sum"), group_flags=("group_flag", "sum"),
        original_mean=("windspeed", "mean"), global_capped_mean=("global_capped", "mean"), group_capped_mean=("group_capped", "mean"))
    policy_summary["global_flag_pct"] = 100 * policy_summary["global_flags"] / policy_summary["days"]
    policy_summary["group_flag_pct"] = 100 * policy_summary["group_flags"] / policy_summary["days"]
    display(policy_summary.round(4))


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Optional challenge C: A data-quality repair sandbox

**Related slides:** 7–10: missing values and types; 24: preprocessing. **Time:** 10 minutes extra.

The cell deliberately introduces an unknown humidity value, a missing value, impossible humidity and an out-of-range temperature.
The real source data are unchanged. Complete numeric conversion with `errors="coerce"`, then median imputation in this sandbox.
Record how many values become missing before filling them. Explain why the imputed values are estimates, not recovered measurements.


In [ ]:
bad = df.copy(deep=True)
bad["humidity_pct"] = bad["humidity_pct"].astype(object)
bad.loc[2, "humidity_pct"] = "unknown"
bad.loc[17, "humidity_pct"] = np.nan
bad.loc[33, "humidity_pct"] = 150
bad.loc[0, "temp"] = 1.2
display(bad.loc[[0, 2, 17, 33], ["date", "humidity_pct", "temp"]])
# TODO: use to_numeric on the humidity column with errors="coerce".
numeric_humidity = None
if numeric_humidity is None:
    print("Complete the numeric-conversion TODO.")
else:
    repaired = bad.copy()
    repaired["humidity_pct"] = numeric_humidity
    for col, minimum, maximum in [("humidity_pct", 0, 100), ("temp", 0, 1)]:
        invalid = repaired[col].notna() & ~repaired[col].between(minimum, maximum)
        repaired.loc[invalid, col] = np.nan
    repair_cols = ["humidity_pct", "temp"]
    print("Missing after coercion and range checks:")
    display(repaired[repair_cols].isna().sum())
    # TODO: fill missing values in these columns using their column medians.
    filled = None
    if filled is None:
        print("Complete the imputation TODO.")
    else:
        repaired[repair_cols] = filled
        display(repaired.loc[[0, 2, 17, 33], ["date", "humidity_pct", "temp"]])
        assert repaired[repair_cols].notna().all().all()
        assert repaired["humidity_pct"].between(0, 100).all()
        assert repaired["temp"].between(0, 1).all()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Optional challenge D: Preprocessing without target leakage

**Related slides:** 24: encoding and scaling, extended to a predictive model. **Time:** 15 minutes extra.

Build an illustrative Ridge regression pipeline for daily rentals.
Use the first 80% of dates for training and the remaining 20% for testing.
Complete the split point and select the predictor list shown below.
Standardise numerical predictors, one-hot encode categories, and fit the whole pipeline only on the training period.
Compare test MAE with a baseline that always predicts the training-period mean.
Explain why a random time split and the same-day component counts could overstate performance.

This example uses recorded weather on held-out days. An operational forecast would need weather forecasts or otherwise
available weather information; its performance may differ.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

# TODO: choose the chronological split point and these seven predictor columns:
# temp, humidity_pct, windspeed, year, season, weather, day_type.
split_point = None
predictor_cols = None
if split_point is None or predictor_cols is None:
    print("Complete the split-point and predictor-list TODOs.")
else:
    assert not {"rentals", "casual", "registered", "record_id"}.intersection(predictor_cols)
    train = df.iloc[:split_point].copy()
    test = df.iloc[split_point:].copy()
    assert train["date"].max() < test["date"].min()
    numerical = ["temp", "humidity_pct", "windspeed", "year"]
    categorical = ["season", "weather", "day_type"]
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), numerical),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])
    model = Pipeline([("preprocessing", preprocessing), ("regression", Ridge(alpha=10))])
    model.fit(train[predictor_cols], train["rentals"])
    predictions = model.predict(test[predictor_cols])
    baseline_predictions = np.full(len(test), train["rentals"].mean())
    print(f"Training days: {len(train)}; test days: {len(test)}")
    print("Training dates:", train["date"].min().date(), "to", train["date"].max().date())
    print("Test dates:", test["date"].min().date(), "to", test["date"].max().date())
    display(pd.DataFrame({"test_MAE": {
        "Training-mean baseline": mean_absolute_error(test["rentals"], baseline_predictions),
        "Ridge pipeline": mean_absolute_error(test["rentals"], predictions),
    }}).round(2))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(test["date"], test["rentals"], color=BLUE, alpha=.65, label="Observed rentals")
    ax.plot(test["date"], predictions, color=TEAL, alpha=.8, label="Ridge predictions")
    ax.set(title="Predictions over the held-out period", xlabel="Date", ylabel="Daily rentals")
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


### Your interpretation

Write 2–3 sentences answering the questions above.

*Your answer:*


## Wrap-up

Write a short EDA conclusion with:

- A finding supported by a table or plot, including its units or group sizes.
- A data issue or coverage limitation worth investigating.
- A preprocessing decision and the reason for it.

The download cache is `bike_sharing_day.csv` and the automatic report is `bike_sharing_sweetviz.html`.
In Colab, use the Files panel to download a generated file if you want to keep it.

### References

- [UCI Bike Sharing data and variable definitions](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)
- [Sweetviz documentation](https://github.com/fbdesignpro/sweetviz)
- [pandas descriptive statistics](https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics)
- [Matplotlib boxplot documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.boxplot.html)
- [scikit-learn common pitfalls: preprocessing and leakage](https://scikit-learn.org/stable/common_pitfalls.html)


In [ ]:
# The core exercises and optional repair/capping demonstrations keep the source intact.
pd.testing.assert_frame_equal(raw, raw_snapshot)
print("Original downloaded data remain unchanged.")
